## Loading Raw Bronze Files into the Postgres Data Warehouse

### Recap: What We're Building

We've extracted data from four sources into MinIO's Raw Bronze layer as Parquet files. Now we need to get that data into a **queryable `bronze` schema inside our Postgres warehouse**, so dbt (and eventually Silver/Gold models) can build on top of it with plain SQL.

Two loading strategies, matching how each source was extracted:
| Pattern | Applies to | Core idea | Idempotency mechanism |
|---|---|---|---|
| **A — Full Snapshot Replace** | `geolocation`, `translations`, `customers`, `products`, `sellers` | Each new file *is* the entire current state. We only ever want the **latest** one. | `TRUNCATE` + reload — running twice on the same file gives the same result, so duplicates are structurally impossible. |
| **B — Incremental Append with File Tracking** | `orders`, `order_items`, `order_payments`, `order_reviews` | Each new file contains **only new rows**, never seen before. A small control table remembers which file keys were already loaded; we simply skip files we've already processed. |


Let's build and test both, one piece at a time, before wrapping everything into reusable classes.

## Why two patterns, not one?

If we tried to use ONE strategy for everything, we'd either:
- re-append the *same* customers/products every run (Pattern B logic on snapshot data → duplicates), or
- throw away order history every run (Pattern A logic on incremental data → data loss).

So the first job of the pipeline is simply: **look at the source, decide which pattern it uses.**
That decision is a config, not code — see the `SOURCES` dict below.

### Setup — imports and environment

This cell loads the Python libraries the rest of the notebook depends on, and pulls in `CONFIG` from `config_loader.py` (a small helper that centralizes credentials and connection settings so they're never hard-coded in the notebook itself).

- `%load_ext autoreload` / `%autoreload 2` — automatically reloads any local module (like `config_loader`) if you edit it, without needing to restart the kernel.
- `sys.path.append(...)` — makes the parent directory importable, since `config_loader.py` lives one level above this notebook.

In [ ]:
# 1. Enable autoreload first
%load_ext autoreload
%autoreload 2

from sqlalchemy import create_engine, text
import pandas as pd
import io
import os
from pprint import pprint

import sys
sys.path.append(os.path.dirname(os.getcwd()))  # Add the parent directory to the sys.path to import config_loader.py. Since the config_loader.py is in the parent directory, we need to add the parent directory to the sys.path so that Python can find it.
from config_loader import CONFIG # Make sure you can access CONFIG variable .


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Connecting to MinIO (S3) and Postgres

Two client objects are created here, one per system:

- `s3` — a `boto3` S3 client pointed at MinIO's `endpoint_url` instead of AWS. Because MinIO speaks the S3 API, every `boto3` S3 call used later (`list_objects_v2`, `get_object`, etc.) works exactly the same as it would against real AWS S3.
- `engine` — a SQLAlchemy engine for the Postgres analytics database, used later by `pandas.to_sql()` and for running raw SQL through `engine.begin()`.

All credentials come from `CONFIG`, not hard-coded values — the same config file governs every connection in this notebook.

In [ ]:
import boto3
from botocore.exceptions import ClientError
import pyarrow.parquet as pq
from config_loader import CONFIG

s3 = boto3.client(
    "s3",
    endpoint_url=CONFIG["storage"]["minio_endpoint"],
    aws_access_key_id=CONFIG["storage"]["minio_access_key"],
    aws_secret_access_key=CONFIG["storage"]["minio_secret_key"],
)

BUCKET = CONFIG['storage']['bronze_bucket']

# --- Postgres connection (SQLAlchemy engine, needed for pandas.to_sql) ---
engine = create_engine(CONFIG['database']['analytics']['analytics_url'])

## The `raw-bronze` bucket structure

Before writing any code, let's look at how the bucket is organized. Every path below the bucket
root follows a **Hive-style partition** convention: `key=value/` folders that tell us something
about the data *without opening a single file*. That's exactly what our functions use above
(`extract_partition`, filename filters, etc.) — the structure IS the metadata.
```text
raw-bronze/
├── api/
│   ├── api_endpoint=geolocation/
│   │   └── *.parquet    (one is tagged _latest.parquet)
│   └── api_endpoint=translations/
│       └── *.parquet    (one is tagged _latest.parquet)
│
├── external_s3/
│   ├── stream_event=order_payments/
│   │   └── *.parquet    (many files, new ones added every extraction run)
│   └── stream_event=order_reviews/
│       └── *.parquet    (many files, new ones added every extraction run)
│
├── metadata/
│   ├── external_table=order_payments/ *.json
│   ├── external_table=order_reviews/ *.json
│   ├── table_name=customers/ *.json
│   ├── table_name=order_items/ *.json
│   ├── table_name=orders/ *.json
│   ├── table_name=products/ *.json
│   └── table_name=sellers/ *.json
│
└── postgres/
    ├── batches/
    │   ├── table_name=orders/
    │   │   └── batch_date=2026-08-24/
    │   │       └── *.parquet  (a new file per run, new rows only)
    │   │
    │   └── table_name=order_items/
    │       └── batch_date=2026-08-24/
    │           └── *.parquet  (a new file per run, new rows only)
    └── snapshots/
        ├── table_name=customers/
        │   └── snapshot_date=2026-08-24/
        │       └── *.parquet (full table copy, one folder per date)
        |
        ├── table_name=products/
        │   └── snapshot_date=2026-08-24/
        │       └── *.parquet  (full table copy, one folder per date)
        │
        └── table_name=sellers/
            └── snapshot_date=2026-08-24/
                └── *.parquet   (full table copy, one folder per date)

```
**Reading the tree, folder by folder:**

| Folder | What it tells us | Which pattern it needs |
|---|---|---|
| `api/api_endpoint=.../` | Where data came from (an API) and which endpoint | **Pattern A** — latest-tagged file |
| `external_s3/stream_event=.../` | An event stream source, no dates in the path | **Pattern B** — incremental, track file names |
| `postgres/batches/table_name=.../batch_date=.../` | A daily extract from Postgres, one folder per day | **Pattern B** — incremental, track file names |
| `postgres/snapshots/table_name=.../snapshot_date=.../` | A full table dump from Postgres, one folder per day | **Pattern A** — latest-partition |
| `metadata/` | Not data — JSON describing *how* each extraction job pulled its source | Not loaded into bronze tables; used for pipeline documentation/debugging |

**The one rule to remember:** the partition folder name tells you whether a file is a *delta*
(`batch_date`, no date at all) or a *full state* (`snapshot_date`, `_latest`). That single fact is
what decides append vs. replace for every table we build in this notebook.

In [ ]:
SOURCES = {
    "bronze.api_geolocation": {
        "pattern": "snapshot_tagged",
        "prefix": "api/api_endpoint=geolocation/",
    },
    "bronze.api_translations": {
        "pattern": "snapshot_tagged",
        "prefix": "api/api_endpoint=translations/",
    },
    "bronze.pg_customers": {
        "pattern": "snapshot_partition",
        "prefix": "postgres/snapshots/table_name=customers/",
    },
    "bronze.pg_products": {
        "pattern": "snapshot_partition",
        "prefix": "postgres/snapshots/table_name=products/",
    },
    "bronze.pg_sellers": {
        "pattern": "snapshot_partition",
        "prefix": "postgres/snapshots/table_name=sellers/",
    },
    "bronze.pg_orders": {
        "pattern": "incremental",
        "prefix": "postgres/batches/table_name=orders/",
    },
    "bronze.pg_order_items": {
        "pattern": "incremental",
        "prefix": "postgres/batches/table_name=order_items/",
    },
    "bronze.stream_order_payments": {
        "pattern": "incremental",
        "prefix": "external_s3/stream_event=order_payments/",
    },
    "bronze.stream_order_reviews": {
        "pattern": "incremental",
        "prefix": "external_s3/stream_event=order_reviews/",
    },
}

**Reading `SOURCES`:** every entry carries exactly two things a loader function needs to know:

- **`pattern`** — which of the three loading strategies applies (`snapshot_tagged`, `snapshot_partition`, or `incremental`). This is what the dispatcher function checks later to decide *how* to load a table.
- **`prefix`** — the S3 path under which that source's files live, i.e. *where* to look.

Everything downstream — `upload_from_s3_v3`, `load_incremental`, etc. — is driven entirely by this dictionary. Adding a new source later means adding one entry here, not writing new pipeline code.

#### CONCEPT 1
##### Let's start with simple scenerio.. (geolocation, translations) always publish a file ending in `_latest.parquet`.Finding the current file is just: **filter the file list for that suffix.**

### This logic, Reading all the parquet files from the specific directory is identical for each implementation. So let's create the function to do so.. 

In [ ]:
prefix = SOURCES["bronze.pg_products"]["prefix"]
paginator = s3.get_paginator("list_objects_v2")
files = []
for page in paginator.paginate(Bucket = BUCKET, Prefix = prefix):
    # print(page['Contents'])
    for obj in page.get('Contents',[]):
        if obj['Key'].endswith(".parquet"):
            files.append((obj["Key"], obj["LastModified"].isoformat()))
print(files)

[('postgres/snapshots/table_name=products/snapshot_date=2026-09-03/snapshot_072059.parquet', '2026-09-03T07:20:59.836000+00:00')]


In [ ]:
def list_files_with_meta(prefix: str):
    """Return list of (key, last_modified) for every parquet file under a prefix."""
    paginator = s3.get_paginator("list_objects_v2")
    files = []
    for page in paginator.paginate(Bucket=BUCKET, Prefix=prefix):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith(".parquet"):
                files.append((obj["Key"], obj["LastModified"].isoformat()))
    return files

In [ ]:
files = list_files_with_meta("api/api_endpoint=geolocation/")
print([(key, last_mod) for key, last_mod in files if key.endswith("_latest.parquet")])
del files


[('api/api_endpoint=geolocation/geolocation_latest.parquet', '2026-09-03T07:20:59.479000+00:00')]


#### SOURCE 1: API (GEOLOCATION AND TRANSLATION)

In [ ]:
prefix = "api/api_endpoint=translations/"

def get_latest_tagged_file(prefix: str) -> tuple:
    """Return the (key, last_modified) tuple for the single file under `prefix`
    whose name ends in `_latest.parquet`.

    Used for Pattern A / 'snapshot_tagged' sources (geolocation, translations),
    where the extractor always (re)writes one file tagged as the current version.
    """
    files = list_files_with_meta(prefix)
    return [(key, last_mod) for key, last_mod in files if key.endswith("_latest.parquet")][0]

geolocation_key = get_latest_tagged_file(prefix)
geolocation_key

('api/api_endpoint=translations/translations_latest.parquet',
 '2026-09-03T07:20:44.744000+00:00')

#### CONCEPT 2

Other sources (customers, products, sellers) publish a full snapshot under a
`snapshot_date=YYYY-MM-DD/` folder each run — sometimes more than one snapshot if changes occur in source . Finding "current"
means: **find the maximum `snapshot_date`, then take the latest file inside that partition.**

In [ ]:
p_c_files = list_files_with_meta("postgres/snapshots/table_name=customers/")
p_c_files

[('postgres/snapshots/table_name=customers/snapshot_date=2026-09-03/snapshot_072100.parquet',
  '2026-09-03T07:21:00.154000+00:00')]

To try out "pick the max timestamp" logic without waiting on real MinIO listings,
here's a hand-written stand-in for what `list_files_with_meta` would return for the
`customers` snapshots — four partitions, one per day:

In [ ]:
files = [
    ("postgres/snapshots/table_name=customers/snapshot_date=2026-08-22/snapshot_091530.parquet", "2026-08-22T09:15:31.120000+00:00"),
    ("postgres/snapshots/table_name=customers/snapshot_date=2026-08-23/snapshot_103245.parquet", "2026-08-23T10:32:46.450000+00:00"),
    ("postgres/snapshots/table_name=customers/snapshot_date=2026-08-24/snapshot_152621.parquet", "2026-08-25T15:26:22.036000+00:00"),
    ("postgres/snapshots/table_name=customers/snapshot_date=2026-08-25/snapshot_071205.parquet", "2026-08-25T07:12:06.210000+00:00"),
]


In [ ]:
#  This is the lambda implementation
l_k,l_t = max(files, key = lambda x: x[1])
print(l_k,l_t)

postgres/snapshots/table_name=customers/snapshot_date=2026-08-24/snapshot_152621.parquet 2026-08-25T15:26:22.036000+00:00


In [ ]:
# IF lambda function is confusing use this insted. It's a same thing
latest_key = None
latest_timestamp = None

for key, timestamp in files:
    if latest_timestamp is None or timestamp > latest_timestamp:
        latest_key = key
        latest_timestamp = timestamp
print((latest_key,latest_timestamp))

('postgres/snapshots/table_name=customers/snapshot_date=2026-08-24/snapshot_152621.parquet', '2026-08-25T15:26:22.036000+00:00')


### Now the  function implementation

Wraps the same two lines around a real MinIO listing.

In [ ]:
def get_latest_snapshot_partition_files(prefix: str) -> tuple:
    """Return the (key, last_modified) tuple for the most recently modified file
    under `prefix`.

    Used for Pattern A / 'snapshot_partition' sources (customers, products, sellers),
    where each run writes a full table dump into a new `snapshot_date=.../` folder.
    Picking the max `LastModified` across ALL files under the prefix is equivalent to
    picking the newest snapshot partition, without needing to parse dates out of the
    path ourselves.
    """
    files = list_files_with_meta(prefix)
    if not files:
        return []
    latest_key, latest_timestamp = max(files, key=lambda x: x[1])
    return (latest_key, latest_timestamp)

In [ ]:
latest_key_sellers = get_latest_snapshot_partition_files('postgres/snapshots/table_name=customers/')
latest_key_sellers

('postgres/snapshots/table_name=customers/snapshot_date=2026-09-03/snapshot_072100.parquet',
 '2026-09-03T07:21:00.154000+00:00')

#### Now we need to read the content of these latest modified files and load them into our table in postgres_anlytics, bronze schema.

### Reading parquate files and converting them in to pandas df..

In [ ]:
body = s3.get_object(Bucket = BUCKET, Key = geolocation_key[0])['Body'].read()
# pprint(body.get('Body').read())
df = pd.read_parquet(io.BytesIO(body), engine='pyarrow')
display(df)
del body, df

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor
...,...,...
66,flores,flowers
67,artes_e_artesanato,arts_and_craftmanship
68,fraldas_higiene,diapers_and_hygiene
69,fashion_roupa_infanto_juvenil,fashion_childrens_clothes


## Step 2 — Create bronze tables from an explicit `.sql` file

Instead of letting pandas guess column types, we write the `CREATE TABLE` statements ourselves in
`create_bronze_tables.sql` and run that file against Postgres. This is the "single source of
truth" for what each bronze table looks like.

In [ ]:
sql_script = """
CREATE SCHEMA IF NOT EXISTS bronze;

CREATE TABLE IF NOT EXISTS bronze.api_geolocation (
    geolocation_zip_code_prefix VARCHAR(10),
    geolocation_lat             DOUBLE PRECISION,
    geolocation_lng             DOUBLE PRECISION,
    geolocation_city            VARCHAR(100),
    geolocation_state           VARCHAR(2),
    _source_file                TEXT,
    _ingested_at                TIMESTAMP DEFAULT now()
);

CREATE TABLE IF NOT EXISTS bronze.api_translations (
    product_category_name          VARCHAR(100),
    product_category_name_english  VARCHAR(100),
    _source_file                   TEXT,
    _ingested_at                   TIMESTAMP DEFAULT now()
);

CREATE TABLE IF NOT EXISTS bronze.stream_order_payments (
    order_id             VARCHAR(64),
    payment_sequential   INTEGER,
    payment_type         VARCHAR(30),
    payment_installments INTEGER,
    payment_value        NUMERIC(12,2),
    _source_file         TEXT,
    _ingested_at         TIMESTAMP DEFAULT now()
);

CREATE TABLE IF NOT EXISTS bronze.stream_order_reviews (
    review_id               VARCHAR(64),
    order_id                VARCHAR(64),
    review_score            SMALLINT,
    review_comment_title    TEXT,
    review_comment_message  TEXT,
    review_creation_date    TIMESTAMP,
    review_answer_timestamp TIMESTAMP,
    _source_file            TEXT,
    _ingested_at             TIMESTAMP DEFAULT now()
);

CREATE TABLE IF NOT EXISTS bronze.pg_orders (
    order_id                       VARCHAR(64),
    customer_id                    VARCHAR(64),
    order_status                   VARCHAR(20),
    order_purchase_timestamp       TIMESTAMP,
    order_approved_at              TIMESTAMP,
    order_delivered_carrier_date   TIMESTAMP,
    order_delivered_customer_date  TIMESTAMP,
    order_estimated_delivery_date  TIMESTAMP,
    _source_file                   TEXT,
    _ingested_at                   TIMESTAMP DEFAULT now()
);

CREATE TABLE IF NOT EXISTS bronze.pg_order_items (
    order_id             VARCHAR(64),
    order_item_id        INTEGER,
    product_id           VARCHAR(64),
    seller_id            VARCHAR(64),
    shipping_limit_date  TIMESTAMP,
    price                NUMERIC(12,2),
    freight_value        NUMERIC(12,2),
    _source_file         TEXT,
    _ingested_at         TIMESTAMP DEFAULT now()
);

CREATE TABLE IF NOT EXISTS bronze.pg_customers (
    customer_id               VARCHAR(64),
    customer_unique_id        VARCHAR(64),
    customer_zip_code_prefix  VARCHAR(10),
    customer_city             VARCHAR(100),
    customer_state            VARCHAR(2),
    _source_file              TEXT,
    _ingested_at              TIMESTAMP DEFAULT now()
);

CREATE TABLE IF NOT EXISTS bronze.pg_products (
    product_id                   VARCHAR(64),
    product_category_name        VARCHAR(100),
    product_name_lenght          NUMERIC,
    product_description_lenght   NUMERIC,
    product_photos_qty           NUMERIC,
    product_weight_g             NUMERIC(10,2),
    product_length_cm            NUMERIC(10,2),
    product_height_cm            NUMERIC(10,2),
    product_width_cm             NUMERIC(10,2),
    _source_file                 TEXT,
    _ingested_at                 TIMESTAMP DEFAULT now()
);

CREATE TABLE IF NOT EXISTS bronze.pg_sellers (
    seller_id               VARCHAR(64),
    seller_zip_code_prefix  VARCHAR(10),
    seller_city             VARCHAR(100),
    seller_state            VARCHAR(2),
    _source_file            TEXT,
    _ingested_at            TIMESTAMP DEFAULT now()
);
"""

with open("create_bronze_tables.sql", "w") as f:
    f.write(sql_script)

print("create_bronze_tables.sql written.")

create_bronze_tables.sql written.


**Design choices worth noting:**

- Every column here is deliberately **loosely typed** (`VARCHAR`, `NUMERIC`, `TEXT`) with no `NOT NULL`, `PRIMARY KEY`, or `CHECK` constraints. Bronze's job is to land raw data faithfully — a strict constraint here would let one bad row fail an entire load. Real validation belongs in Silver/Gold.
- Every table carries two lineage columns: `_source_file` (which S3 key the row came from) and `_ingested_at` (when it was loaded). These make debugging and reprocessing possible without needing external logs.
- The DDL is written to a `.sql` file and executed separately (next cell), rather than letting `pandas.to_sql()` auto-create the tables — this avoids pandas' type inference guessing wrong (a classic bug: a numeric-looking ID column silently becoming an `INTEGER` and losing leading zeros).

In [ ]:
def run_sql_file(filepath: str):
    """Execute every statement in a .sql file against Postgres in one transaction."""
    with open(filepath, "r") as f:
        script = f.read()
    with engine.begin() as conn:
        conn.execute(text(script))
    print(f"Executed {filepath}")

run_sql_file("create_bronze_tables.sql")

Executed create_bronze_tables.sql



For snapshot sources, we don't append — we **replace**. `TRUNCATE` empties the table, then we
insert the current snapshot. Run it once or run it ten times with the same file: the table ends up
identical either way. That's what makes it duplicate-proof.

In contrast to the tagged/snapshot sources above, incremental sources like
`order_payments` accumulate **many** files over time — one per extraction run, each
containing only new rows. Reading "the current state" here isn't one file, it's every
file we haven't loaded yet. Let's see what pulling *all* of them into one DataFrame
looks like (we'll narrow this down to "only the new ones" once the control table is
in place).

In [ ]:
files_payments = list_files_with_meta(SOURCES['bronze.pg_orders']['prefix'])
# pprint(files_payments)
keys = [obj[0] for obj in files_payments]
print(keys)


['postgres/batches/table_name=orders/batch_date=2026-09-03/batch_072100.parquet']


In [ ]:
def read_keys_to_df(keys: list[str]) -> pd.DataFrame:
    """Download each parquet file in `keys` from MinIO and concatenate them into
    a single DataFrame.

    A `_source_file` column is stamped onto every row so we can always trace a
    loaded row back to the exact file it came from (useful for debugging and for
    the incremental control table below).
    """
    frames = []

    for key in keys:
        body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()
        df = pd.read_parquet(io.BytesIO(body), engine="pyarrow")
        df["_source_file"] = key
        frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [ ]:
df = read_keys_to_df(keys)
display(df)
del df

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,_source_file
0,2e7a8482f6fb09756ca50c10d7bfc047,08c5351a6aca1c1589a38f244edeee9d,shipped,2016-09-04 21:15:19,2016-10-07 13:18:03,2016-10-18 13:14:51,NaT,2016-10-20,postgres/batches/table_name=orders/batch_date=...
1,e5fa5a7210941f7d56d0208e4e071d35,683c54fc24d40ee9f8a6fc179fd9856c,canceled,2016-09-05 00:15:34,2016-10-07 13:17:15,NaT,NaT,2016-10-28,postgres/batches/table_name=orders/batch_date=...
2,809a282bbd5dbcabb6f2f724fca862ec,622e13439d6b5a0b486c435618b2679e,canceled,2016-09-13 15:24:19,2016-10-07 13:16:46,NaT,NaT,2016-09-30,postgres/batches/table_name=orders/batch_date=...
3,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,postgres/batches/table_name=orders/batch_date=...
4,71303d7e93b399f5bcd537d124c0bcfa,b106b360fe2ef8849fbbd056f777b4d5,canceled,2016-10-02 22:07:52,2016-10-06 15:50:56,NaT,NaT,2016-10-25,postgres/batches/table_name=orders/batch_date=...
...,...,...,...,...,...,...,...,...,...
19105,e7c93d513c10924fe1f15a0838e4ce95,3f6ae99881c82dfb4cfbb0f95e17d157,delivered,2017-08-01 23:41:09,2017-08-02 00:35:16,2017-08-04 19:57:54,2017-08-07 21:16:49,2017-08-14,postgres/batches/table_name=orders/batch_date=...
19106,b8f158c49d2b09fe06025a4c5dc346f7,e282126399eecb8506f61840a0d65c15,delivered,2017-08-02 00:24:21,2017-08-02 00:35:17,2017-08-02 20:17:45,2017-08-04 20:42:04,2017-08-17,postgres/batches/table_name=orders/batch_date=...
19107,854b1633927e30873792f06e568ed766,4dd6d98ac5cadd43fde7a222269c90ad,delivered,2017-08-02 01:07:28,2017-08-02 01:23:32,2017-08-03 18:49:44,2017-08-09 20:12:32,2017-08-24,postgres/batches/table_name=orders/batch_date=...
19108,e3b442a0dbd0eab458fb4d4abbead49e,9eb9aee89944d31410282c1437ea8c63,delivered,2017-08-02 03:47:44,2017-08-02 21:23:30,2017-08-04 18:13:01,2017-08-10 20:09:34,2017-08-24,postgres/batches/table_name=orders/batch_date=...


## Step 3 — Load each DataFrame into its bronze table

The table (and its column types) already exist — created from our `.sql` file. `to_sql` with
`if_exists='append'` just inserts rows into that existing structure; it is **not** deciding the
schema anymore, Postgres already enforces it.

### A manual walk-through, end-to-end, for one table

Before wrapping every step into reusable functions, let's do the whole load manually for `bronze.api_translations`: list its files, read them into a DataFrame, split `"schema.table"` into its two parts (needed because `to_sql()` takes `schema=` and the table name as separate arguments), then truncate and load. Seeing every step spelled out once makes the abstracted functions below much easier to follow.

In [ ]:
table_name = 'bronze.api_translations'
prefix = SOURCES[table_name]['prefix']
# prefix
files = list_files_with_meta(prefix)
keys = [obj[0] for obj in files]
# keys
df = read_keys_to_df(keys)

schema_name, plain_table = table_name.split(".")

# print(schema_name, plain_table)


with engine.begin() as conn:
    conn.execute(text(f"TRUNCATE TABLE {table_name};"))
    # When you pass a DataFrame to to_sql() without batching, SQLAlchemy/Pandas can perform a large number of individual inserts, especially for a large DataFrame.
    # Use method="multi" and a reasonable chunksize:
    df.to_sql(plain_table, conn, schema=schema_name, if_exists="append", index=False, chunksize=5000, method="multi" )
print(f"[{table_name}] replaced with {len(df)} rows from {len(keys)} file(s).")


[bronze.api_translations] replaced with 71 rows from 1 file(s).


## Two ways to get a DataFrame into Postgres

Both functions do the same job (`TRUNCATE` then reload), but the **load step** works completely
differently under the hood.

### `to_sql(..., method="multi", chunksize=50000)` — v1

Pandas builds actual SQL `INSERT` statements and sends them through the DB-API cursor:

```sql
INSERT INTO bronze.pg_customers (customer_id, customer_city, ...)
VALUES (%s, %s, ...), (%s, %s, ...), ..._up to 50000 rows per batch_;
```

- Each value is bound as a parameter (safe from SQL injection, and each value gets **type-checked
  by the driver** against the column's declared parameter type).
  It goes through the normal SQL parser/planner path — same machinery as any other `INSERT`.
- `chunksize=50000` just controls how many rows go in one `INSERT` statement before starting a new
  one — doesn't change the mechanism, just the batch size.

### `copy_expert(...)` — v2

Postgres has a **separate, non-SQL wire protocol** just for bulk load called `COPY`. `psycopg2`'s
`copy_expert` streams a CSV buffer directly into that protocol:

- No `INSERT` statements are parsed at all. No per-row SQL planning, no parameter binding.
- Postgres reads the stream, splits it on the given format (`CSV` here), and writes rows straight
  into the table's storage — this is the same mechanism `psql \copy` and `pg_dump`/`pg_restore` use.
- This is the **fastest possible way** to bulk-load data into Postgres, by a wide margin.

In [ ]:
def upload_from_s3_v1(table_name: str) -> None:
    prefix = SOURCES[table_name]['prefix']
    if not prefix:
        raise ValueError(f"No source found for {table_name}")

    keys = [x[0] for x in list_files_with_meta(prefix)]
    if not keys:
        print(f"[{table_name}] No files found.")
        return

    df = read_keys_to_df(keys)
    schema, table = table_name.split(".", 1)

    with engine.begin() as conn:
        conn.execute(text(f"TRUNCATE TABLE {table_name}"))
        df.to_sql(table, conn, schema=schema, if_exists="append", index=False,chunksize=50000, method="multi")

    print(f"[{table_name}] replaced with {len(df)} rows from {len(keys)} file(s).")

In [ ]:
upload_from_s3_v1('bronze.api_translations')

[bronze.api_translations] replaced with 71 rows from 1 file(s).


In [ ]:
def upload_from_s3_v2(table_name: str) -> None:
    prefix = SOURCES[table_name]['prefix']
    if not prefix:
        raise ValueError(f"No source found for {table_name}")

    keys = [x[0] for x in list_files_with_meta(prefix)]
    if not keys:
        return

    df = read_keys_to_df(keys)
    cols = list(df.columns)
    buf = io.StringIO()
    df.to_csv(buf, index=False, header=False, na_rep="\\N")
    buf.seek(0)

    with engine.begin() as conn:
        conn.execute(text(f"TRUNCATE TABLE {table_name}"))
        raw = conn.connection.driver_connection
        with raw.cursor() as cur:
            cur.copy_expert(
                f"COPY {table_name} ({','.join(cols)}) FROM STDIN WITH CSV NULL '\\N'",
                buf,
            )

    print(f"[{table_name}] replaced with {len(df)} rows from {len(keys)} file(s).")

In [ ]:
upload_from_s3_v2('bronze.api_geolocation')

[bronze.api_geolocation] replaced with 1000163 rows from 1 file(s).


`replace_table_from_s3_v2` above is hard-coded to look up its own prefix from `SOURCES`
and always truncates. `load_df_to_postgres` generalizes it: it takes an already-built
DataFrame and a `truncate` flag, so the same function can be reused for both
**truncate + replace** (snapshot patterns) and **append-only** (incremental pattern) —
which is exactly what `replace_table_from_s3_v3` further down needs.

In [ ]:
def load_df_to_postgres(df: pd.DataFrame, table_name: str, truncate: bool = True) -> None:
    """Load a DataFrame into a Postgres table using COPY (fast bulk load).

    Parameters
    ----------
    df : DataFrame to load. Column names must match the target table's columns.
    table_name : "schema.table" — e.g. "bronze.pg_customers".
    truncate : if True, empties the table before loading (use for full-snapshot replace).
               Set False for incremental appends.
    """
    if df.empty:
        print(f"[{table_name}] DataFrame is empty, nothing to load.")
        return

    cols = list(df.columns)
    col_list = ",".join(f'"{c}"' for c in cols)  # quoted, safe against reserved words

    buf = io.StringIO()
    df.to_csv(buf, index=False, header=False, na_rep="\\N")
    buf.seek(0)

    with engine.begin() as conn:
        if truncate:
            conn.execute(text(f"TRUNCATE TABLE {table_name}"))

        raw = conn.connection.driver_connection
        with raw.cursor() as cur:
            cur.copy_expert(
                f"COPY {table_name} ({col_list}) FROM STDIN WITH CSV NULL '\\N'",
                buf,
            )

    print(f"[{table_name}] loaded {len(df)} rows.(truncate={truncate}).")

#### LET'S PUT ALL THE LOGIC TOGETHER
1. First list all the files available in s3 for each table.
2. For Snapshot, find out the latest one.
3. Read data from parquet to df
4. Upload it to the postgres database

In [ ]:
def upload_from_s3(table_name: str) -> None:
    prefix = SOURCES[table_name]['prefix']
    pattern = SOURCES[table_name]['pattern']

    if pattern == "snapshot_tagged":
        key,modified_at = get_latest_tagged_file(prefix)
        print(f"Table: {table_name} \n  Key: {key}")
        df = read_keys_to_df([key])
        load_df_to_postgres(df, table_name,truncate=True)
        del df

    if pattern == "snapshot_partition":
        key,modified_at = get_latest_snapshot_partition_files(prefix)
        print(f"Table: {table_name} \n  Key: {key}")
        df = read_keys_to_df([key])
        load_df_to_postgres(df, table_name,truncate=True)
        del df

    if pattern == "incremental":
        # load_incremental(table_name=table_name)        
        print(f"Skipping Incremental load for {table_name} for now")

In [ ]:
# tables = list(SOURCES.keys())
# for tab in tables:
#     upload_from_s3(tab)

### CONCEPT 4

For incremental sources (orders, order_items, payments, reviews), every new file has genuinely new
rows. So instead of tracking rows, we just remember **which file names we already loaded**, and
skip them next time — a simple "already seen" set is the entire mechanism. To implement that we need to create a control table taht store the information about the "already seen" set of data files.


The "already loaded" set now lives permanently in a Postgres control table
(`bronze._processed_files`), so it survives between pipeline runs — not just within one script
execution like the demo above.

In [ ]:
def create_control_table():
    """Create the bronze._processed_files control table if it doesn't exist yet.

    This table is what makes Pattern B (incremental append) idempotent: for every
    file we successfully load, we record (table_name, file_key). On the next run
    we simply skip any file_key already present, so re-running the pipeline never
    re-inserts the same rows twice.
    """
    with engine.begin() as conn:
        conn.execute(text("""
            CREATE TABLE IF NOT EXISTS bronze._processed_files (
                table_name  VARCHAR(100) NOT NULL,
                file_key    TEXT NOT NULL,
                loaded_at   TIMESTAMP NOT NULL DEFAULT now(),
                row_count   INTEGER,
                PRIMARY KEY (table_name, file_key)
            );
        """))

create_control_table()

In [ ]:
# Sanity check: right after creation the control table should be empty for every
# table, since nothing has been loaded through it yet.
table_name = "bronze.pg_orders"

with engine.begin() as conn:
    processed = conn.execute(
        text("""
            SELECT file_key
            FROM bronze._processed_files
            WHERE table_name = :table
        """),
        {"table": table_name},
    ).scalars().all()

processed_keys = set(processed)
processed_keys

set()

### Putting the control table to work

Now that `bronze._processed_files` exists (and we've confirmed it starts empty), `load_incremental` below is the function that actually uses it: for a given table, it lists all source files, filters out any `file_key` already present in the control table, loads only the new ones with `COPY`, and records each successfully loaded file back into the control table — inside the same transaction as the load, so a failed load is never mistakenly marked as processed.

In [ ]:
def load_incremental(table_name: str):
    """Load only the files for `table_name` that aren't already in the control table.

    Steps:
    1. List every parquet file under the source's prefix.
    2. Look up which file_keys are already recorded in bronze._processed_files.
    3. For each new file: read it, COPY it straight into the target table (append,
       no truncate), then record the file_key in the control table.

    Because step 3 records the file only after a successful load, and the INSERT
    uses `ON CONFLICT ... DO NOTHING`, re-running this function is always safe —
    already-loaded files are simply skipped.
    """
    prefix = SOURCES[table_name]["prefix"]

    files = list_files_with_meta(prefix)

    with engine.begin() as conn:
        processed_keys = set(
            conn.execute(
                text("""
                    SELECT file_key
                    FROM bronze._processed_files
                    WHERE table_name = :table
                """),
                {"table": table_name},
            ).scalars()
        )

    new_keys = [key for key, _ in files if key not in processed_keys]

    if not new_keys:
        print(f"[{table_name}] no new files to load")

    for key in new_keys:
        df = read_keys_to_df([key])
        cols = list(df.columns)

        buf = io.StringIO()
        df.to_csv(buf, index=False, header=False, na_rep="\\N")
        buf.seek(0)

        with engine.begin() as conn:
            raw = conn.connection.driver_connection

            with raw.cursor() as cur:
                cur.copy_expert(
                    f"COPY {table_name} ({','.join(cols)}) "
                    "FROM STDIN WITH CSV NULL '\\N'",
                    buf,
                )

            conn.execute(
                text("""
                    INSERT INTO bronze._processed_files
                        (table_name, file_key, row_count)
                    VALUES (:t, :k, :n) ON CONFLICT (table_name, file_key) DO NOTHING
                """),
                {"t": table_name, "k": key, "n": len(df)},
            )
        print(f"[{table_name}] loaded {len(df)} rows from {key}")
        del df

### CONCEPT 5 — One function to load any table

We now have every building block: file-finders for both snapshot flavors, a
DataFrame reader, a fast Postgres loader, and an incremental loader backed by the
control table. `replace_table_from_s3_v3` ties them together — given just a table
name, it looks up that table's `pattern` in `SOURCESv2` and calls the right
combination of the functions above.

In [ ]:
def upload_from_s3_v3(table_name: str) -> None:
    """Master entry point: load `table_name` using whichever pattern it needs.

    Looks up `table_name` in SOURCESv2, reads its `pattern`, and dispatches:
      - 'snapshot_tagged'    -> find the file tagged _latest.parquet, truncate + load it
      - 'snapshot_partition' -> find the file in the newest snapshot_date/ partition, truncate + load it
      - 'incremental'        -> hand off to load_incremental(), which appends only new files

    This is the function the final pipeline loop below calls for every source.
    """
    prefix = SOURCES[table_name]['prefix']
    pattern = SOURCES[table_name]['pattern']

    if pattern == "snapshot_tagged":
        key, modified_at = get_latest_tagged_file(prefix)
        print(f"Table: {table_name} \n  Key: {key}")
        df = read_keys_to_df([key])
        load_df_to_postgres(df, table_name, truncate=True)
        del df

    if pattern == "snapshot_partition":
        key, modified_at = get_latest_snapshot_partition_files(prefix)
        print(f"Table: {table_name} \n  Key: {key}")
        df = read_keys_to_df([key])
        load_df_to_postgres(df, table_name, truncate=True)
        del df

    if pattern == "incremental":
        load_incremental(table_name=table_name)
        # print(f"Skipping Incremental load for {table_name} for now")

## Step 4 — Run the full pipeline

With `create_control_table`, `load_incremental`, and `replace_table_from_s3_v3` all
defined, we can finally loop over every source in `SOURCESv2` and load it with the
correct strategy — this is the cell you'd actually schedule to run on each pipeline
run.

In [ ]:
for table in SOURCES.keys():
    print(table)
    upload_from_s3_v3(table)
    print(f"Loading data for {table} complete.")

bronze.api_geolocation
Table: bronze.api_geolocation 
  Key: api/api_endpoint=geolocation/geolocation_latest.parquet


[bronze.api_geolocation] loaded 1000163 rows.(truncate=True).
Loading data for bronze.api_geolocation complete.
bronze.api_translations
Table: bronze.api_translations 
  Key: api/api_endpoint=translations/translations_latest.parquet
[bronze.api_translations] loaded 71 rows.(truncate=True).
Loading data for bronze.api_translations complete.
bronze.pg_customers
Table: bronze.pg_customers 
  Key: postgres/snapshots/table_name=customers/snapshot_date=2026-09-03/snapshot_072100.parquet
[bronze.pg_customers] loaded 99441 rows.(truncate=True).
Loading data for bronze.pg_customers complete.
bronze.pg_products
Table: bronze.pg_products 
  Key: postgres/snapshots/table_name=products/snapshot_date=2026-09-03/snapshot_072059.parquet
[bronze.pg_products] loaded 32951 rows.(truncate=True).
Loading data for bronze.pg_products complete.
bronze.pg_sellers
Table: bronze.pg_sellers 
  Key: postgres/snapshots/table_name=sellers/snapshot_date=2026-09-03/snapshot_072100.parquet
[bronze.pg_sellers] loaded 30